In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

# Function
def extract_frames(video_path, output_folder):

    cap = cv2.VideoCapture(video_path)

    # Video name
    video_name = os.path.basename(video_path).split(".")[0]

    frame_no = 0

    # Check video
    if not cap.isOpened():
        print("Error opening video")
        return

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # Frame name
        frame_name = f"{video_name}_{frame_no}.jpg"

        # Save frame
        cv2.imwrite(
            os.path.join(output_folder, frame_name),
            frame
        )

        frame_no += 1

    cap.release()

    print(f"{frame_no} frames extracted successfully!")


# Video path
video_path = input("Enter video path: ")

# Output folder
output_folder = "../finalRun/frames/extracted"

# Create folders
os.makedirs(output_folder, exist_ok=True)

# Extract frames
extract_frames(video_path, output_folder)

# Enhancement
# Create output folders

input_folder = "../finalRun/frames/extracted"
output_folder = "../finalRun/frames/enhanced"

folders = [
    "1_Grayscale",
    "2_NoiseReduced",
    "3_IlluminationCorrected",
    "4_FinalEnhanced"
]

# Create main folder

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Create subfolders

for folder in folders:

    path = os.path.join(output_folder, folder)

    if not os.path.exists(path):
        os.makedirs(path)

# Create category folders

categories = [
    "AligatorCracking",
    "TransverseCracking",
    "Pothole",
    "Raveling"
]

for category in categories:

    os.makedirs(
        os.path.join(output_folder, "4_FinalEnhanced", category),
        exist_ok=True
    )


# Convert images to grayscale

for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        img = cv2.imread(os.path.join(input_folder, file))

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        cv2.imwrite(
            os.path.join(output_folder, "1_Grayscale", file),
            gray
        )


# Noise reduction and enhancement

frames_folder = "../finalRun/frames/enhanced/1_Grayscale"
noise_output = "../finalRun/frames/enhanced/2_NoiseReduced"

os.makedirs(noise_output, exist_ok=True)

for file in os.listdir(frames_folder):

    if not file.endswith((".jpg", ".png", ".jpeg")):
        continue

    path = os.path.join(frames_folder, file)

    img = cv2.imread(path)

    if img is None:
        continue

    # Median filter

    median = cv2.medianBlur(img, 3)

    # Gaussian blur

    denoised = cv2.GaussianBlur(median, (5,5), 0)

    # CLAHE

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    gray = cv2.cvtColor(denoised, cv2.COLOR_BGR2GRAY)

    enhanced = clahe.apply(gray)

    # Sharpening

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharpened = cv2.filter2D(
        enhanced,
        -1,
        kernel
    )

    # Save output

    cv2.imwrite(
        os.path.join(noise_output, file),
        sharpened
    )


# Illumination correction

illumination_input = "../finalRun/frames/enhanced/2_NoiseReduced"
illumination_output = "../finalRun/frames/enhanced/3_IlluminationCorrected"

os.makedirs(illumination_output, exist_ok=True)


# Contrast stretching function

def contrast_stretch(img):

    min_val = np.min(img)
    max_val = np.max(img)

    if max_val - min_val == 0:
        return img.copy()

    stretched = (img - min_val) * (
        255 / (max_val - min_val)
    )

    return np.clip(stretched, 0, 255).astype(np.uint8)


for file in os.listdir(illumination_input):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(illumination_input, file)

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        contrast_img = contrast_stretch(img)

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8,8)
        )

        illum_corrected = clahe.apply(contrast_img)

        illum_corrected = cv2.GaussianBlur(
            illum_corrected,
            (3,3),
            0
        )

        cv2.imwrite(
            os.path.join(illumination_output, file),
            illum_corrected
        )


# Final enhancement

final_input = "../finalRun/frames/enhanced/3_IlluminationCorrected"
final_output = "../finalRun/frames/enhanced/4_FinalEnhanced"

max_images = 3
count = 0


# Calculate sharpness

def calculate_sharpness(img):

    return cv2.Laplacian(
        img,
        cv2.CV_64F
    ).var()


# Calculate edge strength

def calculate_edge_strength(img):

    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)

    magnitude = np.sqrt(sobelx**2 + sobely**2)

    return np.mean(magnitude)


# Detect blur type

def detect_blur_type(sharpness, edge_strength):

    if sharpness < 50:
        return "Strong Blur"

    elif sharpness < 100:
        return "Moderate Blur"

    elif edge_strength < 20:
        return "Edge Loss"

    else:
        return "Sharp"


# Enhance image

def enhance_image(img, blur_type):

    base = cv2.GaussianBlur(img, (3,3), 0)

    if blur_type == "Strong Blur":

        gaussian = cv2.GaussianBlur(base, (0,0), 2.0)

        enhanced = cv2.addWeighted(
            base,
            1.4,
            gaussian,
            -0.4,
            0
        )

    elif blur_type == "Moderate Blur":

        gaussian = cv2.GaussianBlur(base, (0,0), 1.5)

        enhanced = cv2.addWeighted(
            base,
            1.3,
            gaussian,
            -0.3,
            0
        )

    elif blur_type == "Edge Loss":

        kernel = np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ])

        enhanced = cv2.filter2D(base, -1, kernel)

    else:

        enhanced = cv2.convertScaleAbs(
            base,
            alpha=1.05,
            beta=0
        )

    enhanced = cv2.GaussianBlur(
        enhanced,
        (3,3),
        0
    )

    return enhanced


# Process images

for filename in os.listdir(final_input):

    if filename.lower().endswith((
        '.jpg',
        '.jpeg',
        '.png',
        '.bmp'
    )):

        path = os.path.join(final_input, filename)

        image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            continue

        sharp_before = calculate_sharpness(image)
        edge_before = calculate_edge_strength(image)

        blur_type = detect_blur_type(
            sharp_before,
            edge_before
        )

        enhanced = enhance_image(image, blur_type)

        # Save category folders

        lower_name = filename.lower()

        if "allig" in lower_name:
            category_folder = "AligatorCracking"

        elif "trans" in lower_name:
            category_folder = "TransverseCracking"

        elif "poth" in lower_name:
            category_folder = "Pothole"

        elif "ravel" in lower_name:
            category_folder = "Raveling"

        else:
            continue

        save_path = os.path.join(
            final_output,
            category_folder,
            filename
        )

        cv2.imwrite(save_path, enhanced)

        # Final display only

        if count < max_images:

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            plt.imshow(image, cmap='gray')
            plt.title("Input")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(enhanced, cmap='gray')
            plt.title("Final Enhanced")
            plt.axis("off")

            plt.subplot(1,3,3)

            hist = cv2.calcHist(
                [enhanced],
                [0],
                None,
                [256],
                [0,256]
            )

            plt.plot(hist)
            plt.title("Histogram")

            plt.tight_layout()
            plt.show()

            count += 1

print("Enhancement completed successfully!")

# Segmentation
# Create segmentation folders

segmented_output = "../finalRun/frames/segmented"

folders = [
    "AlligatorCracking",
    "Pothole",
    "Raveling",
    "TransverseCracking"
]

if not os.path.exists(segmented_output):
    os.makedirs(segmented_output)

for folder in folders:

    path = os.path.join(segmented_output, folder)

    if not os.path.exists(path):
        os.makedirs(path)


# Alligator cracking segmentation

input_folder = "../finalRun/frames/enhanced/4_FinalEnhanced/AligatorCracking"
output_folder = "../finalRun/frames/segmented/AlligatorCracking"

max_show = 3
count = 0


# Segment cracks
def segment_alligator(gray):

    # Light smoothing
    blur = cv2.GaussianBlur(gray, (3,3), 0)

    # Blackhat enhancement
    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (9,9)
    )

    blackhat = cv2.morphologyEx(
        blur,
        cv2.MORPH_BLACKHAT,
        kernel
    )

    # Binary threshold
    _, binary = cv2.threshold(
        blackhat,
        20,
        255,
        cv2.THRESH_BINARY
    )

    # Remove tiny noise
    open_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3,3)
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        open_kernel,
        iterations=1
    )

    # Moderate connection
    close_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5,5)
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        close_kernel,
        iterations=1
    )

    return binary


# Merge nearby boxes
def merge_boxes(boxes, distance=35):

    merged_boxes = []

    while boxes:

        x1, y1, x2, y2 = boxes.pop(0)

        merged = True

        while merged:

            merged = False

            remove_indices = []

            for i, (xx1, yy1, xx2, yy2) in enumerate(boxes):

                if (
                    abs(xx1 - x2) < distance or
                    abs(x1 - xx2) < distance
                ) and (
                    abs(yy1 - y2) < distance or
                    abs(y1 - yy2) < distance
                ):

                    x1 = min(x1, xx1)
                    y1 = min(y1, yy1)
                    x2 = max(x2, xx2)
                    y2 = max(y2, yy2)

                    remove_indices.append(i)

                    merged = True

            for index in sorted(remove_indices, reverse=True):
                boxes.pop(index)

        merged_boxes.append([x1, y1, x2, y2])

    return merged_boxes


# Detect alligator regions
def detect_alligator_regions(image, mask):

    output = image.copy()

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []

    for c in contours:

        area = cv2.contourArea(c)

        if 100 < area < 2500:

            x, y, w, h = cv2.boundingRect(c)

            aspect_ratio = w / float(h + 1e-5)

            if 0.4 < aspect_ratio < 3.5:

                boxes.append([x, y, x+w, y+h])

    merged_boxes = merge_boxes(boxes)

    for box in merged_boxes:

        x1, y1, x2, y2 = box

        width = x2 - x1
        height = y2 - y1

        area = width * height

        if 3000 < area < 60000:

            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                (0,255,0),
                3
            )

            cv2.putText(
                output,
                "Alligator Crack",
                (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2
            )

    return output


# Main loop
for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)

        img = cv2.imread(path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        mask = segment_alligator(gray)

        result = detect_alligator_regions(img, mask)

        cv2.imwrite(
            os.path.join(output_folder, file),
            result
        )


# Pothole detection

input_folder = "../finalRun/frames/enhanced/4_FinalEnhanced/Pothole"
output_folder = "../finalRun/frames/segmented/Pothole"


# Detect potholes
def detect_pothole(gray, output, mask):

    blur = cv2.GaussianBlur(gray, (7,7), 0)

    _, th = cv2.threshold(
        blur,
        95,
        255,
        cv2.THRESH_BINARY_INV
    )

    th = cv2.bitwise_and(th, th, mask=mask)

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_OPEN,
        np.ones((9,9), np.uint8)
    )

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_CLOSE,
        np.ones((15,15), np.uint8)
    )

    contours, _ = cv2.findContours(
        th,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        area = cv2.contourArea(c)

        if area < 1500:
            continue

        x, y, w, h = cv2.boundingRect(c)

        if w / (h + 1e-5) > 3.5:
            continue

        cv2.rectangle(
            output,
            (x,y),
            (x+w,y+h),
            (0,0,255),
            2
        )

        cv2.putText(
            output,
            "Pothole",
            (x,y-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,0,255),
            2
        )

    return output


# Main loop
for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)

        img = cv2.imread(path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        mask = np.ones_like(gray, dtype=np.uint8) * 255

        output = img.copy()

        result = detect_pothole(gray, output, mask)

        cv2.imwrite(
            os.path.join(output_folder, file),
            result
        )


# Raveling detection

input_folder = "../finalRun/frames/enhanced/4_FinalEnhanced/Raveling"
output_folder = "../finalRun/frames/segmented/Raveling"


# Apply enhancement
def apply_enhancement(img):

    smoothed = cv2.blur(img, (3, 3))

    img_float = smoothed.astype(np.float32)

    constant = 100 / np.log(
        1 + np.max(img_float)
    )

    log_img = constant * np.log(
        1 + img_float
    )

    log_img = np.array(
        log_img,
        dtype=np.uint8
    )

    low = np.min(log_img)
    high = np.max(log_img)

    if high <= low:
        return log_img

    stretched = cv2.convertScaleAbs(
        log_img,
        alpha=(255.0 / (high - low)),
        beta=-(low * 255.0 / (high - low))
    )

    return stretched


# Extract raveling
def extract_damage(enhanced_img):

    gaussian = cv2.GaussianBlur(
        enhanced_img,
        (5, 5),
        0
    )

    binary = cv2.adaptiveThreshold(
        gaussian,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15,
        3
    )

    struct_element = np.ones(
        (3, 3),
        np.uint8
    )

    refined_mask = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        struct_element,
        iterations=2
    )

    return refined_mask


# Detect raveling
def detect_raveling(image, mask):

    output = image.copy()

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        if cv2.contourArea(c) > 150:

            x, y, w, h = cv2.boundingRect(c)

            cv2.rectangle(
                output,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )

            cv2.putText(
                output,
                "Raveling",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    return output


# Main loop
for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)

        img = cv2.imread(path)

        if img is None:
            continue

        gray = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2GRAY
        )

        enhanced = apply_enhancement(gray)

        mask = extract_damage(enhanced)

        result = detect_raveling(img, mask)

        cv2.imwrite(
            os.path.join(output_folder, file),
            result
        )


print("Segmentation completed successfully!")

In [ ]:
import cv2
import os
import numpy as np


# Function
def extract_frames(video_path, output_folder):

    cap = cv2.VideoCapture(video_path)

    video_name = os.path.basename(video_path).split(".")[0]

    frame_no = 0

    if not cap.isOpened():
        print("Error opening video")
        return

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_name = f"{video_name}_{frame_no}.jpg"

        cv2.imwrite(
            os.path.join(output_folder, frame_name),
            frame
        )

        frame_no += 1

    cap.release()

    print(f"{frame_no} frames extracted successfully!")


# Video path
video_path = input("Enter video path: ")


# Folder paths
extracted_folder = "../finalRun/frames/extracted"

enhanced_folder = "../finalRun/frames/enhanced"

final_enhanced_folder = "../finalRun/frames/enhanced/4_FinalEnhanced"

segmented_folder = "../finalRun/frames/segmented"

output_video = "../finalRun/final_segmented_video.mp4"


# Create folders
os.makedirs(extracted_folder, exist_ok=True)

os.makedirs(enhanced_folder, exist_ok=True)

os.makedirs(final_enhanced_folder, exist_ok=True)

os.makedirs(segmented_folder, exist_ok=True)


# Extract frames
extract_frames(video_path, extracted_folder)


# Enhancement function
def enhance_image(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    median = cv2.medianBlur(gray, 3)

    blur = cv2.GaussianBlur(
        median,
        (5,5),
        0
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    enhanced = clahe.apply(blur)

    kernel = np.array([
        [0,-1,0],
        [-1,5,-1],
        [0,-1,0]
    ])

    sharpened = cv2.filter2D(
        enhanced,
        -1,
        kernel
    )

    return sharpened


# Detect alligator cracking
def detect_alligator(frame, gray):

    output = frame.copy()

    blur = cv2.GaussianBlur(gray, (3,3), 0)

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (9,9)
    )

    blackhat = cv2.morphologyEx(
        blur,
        cv2.MORPH_BLACKHAT,
        kernel
    )

    _, binary = cv2.threshold(
        blackhat,
        20,
        255,
        cv2.THRESH_BINARY
    )

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        area = cv2.contourArea(c)

        if 300 < area < 4000:

            x, y, w, h = cv2.boundingRect(c)

            ratio = w / float(h + 1e-5)

            if 0.4 < ratio < 3.5:

                cv2.rectangle(
                    output,
                    (x,y),
                    (x+w,y+h),
                    (0,255,0),
                    2
                )

                cv2.putText(
                    output,
                    "Alligator Crack",
                    (x,y-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0,255,0),
                    2
                )

    return output


# Detect potholes
def detect_pothole(frame, gray):

    output = frame.copy()

    blur = cv2.GaussianBlur(gray, (7,7), 0)

    _, th = cv2.threshold(
        blur,
        95,
        255,
        cv2.THRESH_BINARY_INV
    )

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_OPEN,
        np.ones((9,9), np.uint8)
    )

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_CLOSE,
        np.ones((15,15), np.uint8)
    )

    contours, _ = cv2.findContours(
        th,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        area = cv2.contourArea(c)

        if area > 1500:

            x, y, w, h = cv2.boundingRect(c)

            if w / float(h + 1e-5) < 3.5:

                cv2.rectangle(
                    output,
                    (x,y),
                    (x+w,y+h),
                    (0,0,255),
                    2
                )

                cv2.putText(
                    output,
                    "Pothole",
                    (x,y-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0,0,255),
                    2
                )

    return output


# Detect raveling
def detect_raveling(frame, gray):

    output = frame.copy()

    gaussian = cv2.GaussianBlur(
        gray,
        (5,5),
        0
    )

    binary = cv2.adaptiveThreshold(
        gaussian,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15,
        3
    )

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        if cv2.contourArea(c) > 200:

            x, y, w, h = cv2.boundingRect(c)

            cv2.rectangle(
                output,
                (x,y),
                (x+w,y+h),
                (255,255,0),
                2
            )

            cv2.putText(
                output,
                "Raveling",
                (x,y-5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255,255,0),
                2
            )

    return output


# Detect transverse cracking
def detect_transverse(frame, gray):

    output = frame.copy()

    edges = cv2.Canny(
        gray,
        50,
        150
    )

    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi/180,
        threshold=80,
        minLineLength=100,
        maxLineGap=20
    )

    if lines is not None:

        for line in lines:

            x1, y1, x2, y2 = line[0]

            angle = abs(
                np.degrees(
                    np.arctan2(y2-y1, x2-x1)
                )
            )

            if angle > 70:

                cv2.line(
                    output,
                    (x1,y1),
                    (x2,y2),
                    (255,0,255),
                    3
                )

                cv2.putText(
                    output,
                    "Transverse Crack",
                    (x1,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255,0,255),
                    2
                )

    return output


# Read video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))


# Video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)


# Process video
frame_no = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_name = f"{os.path.basename(video_path).split('.')[0]}_{frame_no}.jpg"

    cv2.imwrite(
        os.path.join(extracted_folder, frame_name),
        frame
    )

    enhanced = enhance_image(frame)

    cv2.imwrite(
        os.path.join(final_enhanced_folder, frame_name),
        enhanced
    )

    result = frame.copy()

    result = detect_alligator(
        result,
        enhanced
    )

    result = detect_pothole(
        result,
        enhanced
    )

    result = detect_raveling(
        result,
        enhanced
    )

    result = detect_transverse(
        result,
        enhanced
    )

    cv2.imwrite(
        os.path.join(segmented_folder, frame_name),
        result
    )

    cv2.imshow(
        "Road Damage Detection",
        result
    )

    writer.write(result)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    frame_no += 1


# Release
cap.release()

writer.release()

cv2.waitKey(0)
cv2.destroyAllWindows()

print("Road damage detection video created successfully!")

In [ ]:
import cv2
import os
import numpy as np


# Function
def extract_frames(video_path, output_folder):

    cap = cv2.VideoCapture(video_path)

    video_name = os.path.basename(video_path).split(".")[0]

    frame_no = 0

    if not cap.isOpened():
        print("Error opening video")
        return

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_name = f"{video_name}_{frame_no}.jpg"

        cv2.imwrite(
            os.path.join(output_folder, frame_name),
            frame
        )

        frame_no += 1

    cap.release()

    print(f"{frame_no} frames extracted successfully!")


# Video path
video_path = input("Enter video path: ")


# Folder paths
extracted_folder = "../finalRun/frames/extracted"

enhanced_folder = "../finalRun/frames/enhanced"

final_enhanced_folder = "../finalRun/frames/enhanced/4_FinalEnhanced"

segmented_folder = "../finalRun/frames/segmented"

output_video = "../finalRun/final_segmented_video.mp4"


# Create folders
os.makedirs(extracted_folder, exist_ok=True)

os.makedirs(enhanced_folder, exist_ok=True)

os.makedirs(final_enhanced_folder, exist_ok=True)

os.makedirs(segmented_folder, exist_ok=True)


# Extract frames
extract_frames(video_path, extracted_folder)


# Enhancement function
def enhance_image(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    median = cv2.medianBlur(gray, 3)

    blur = cv2.GaussianBlur(
        median,
        (5,5),
        0
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    enhanced = clahe.apply(blur)

    kernel = np.array([
        [0,-1,0],
        [-1,5,-1],
        [0,-1,0]
    ])

    sharpened = cv2.filter2D(
        enhanced,
        -1,
        kernel
    )

    return sharpened


# Create road mask
def create_road_mask(gray):

    height, width = gray.shape

    mask = np.zeros_like(gray)

    polygon = np.array([[
        (0, height),
        (width, height),
        (int(width*0.75), int(height*0.35)),
        (int(width*0.25), int(height*0.35))
    ]], np.int32)

    cv2.fillPoly(
        mask,
        polygon,
        255
    )

    return mask


# Detect alligator cracking
def detect_alligator(frame, gray):

    output = frame.copy()

    blur = cv2.GaussianBlur(gray, (3,3), 0)

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (9,9)
    )

    blackhat = cv2.morphologyEx(
        blur,
        cv2.MORPH_BLACKHAT,
        kernel
    )

    _, binary = cv2.threshold(
        blackhat,
        25,
        255,
        cv2.THRESH_BINARY
    )

    open_kernel = np.ones((3,3), np.uint8)

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        open_kernel
    )

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        area = cv2.contourArea(c)

        if 800 < area < 6000:

            x, y, w, h = cv2.boundingRect(c)

            ratio = w / float(h + 1e-5)

            if 0.5 < ratio < 3.0:

                cv2.rectangle(
                    output,
                    (x,y),
                    (x+w,y+h),
                    (0,255,0),
                    2
                )

                cv2.putText(
                    output,
                    "Alligator Crack",
                    (x,y-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0,255,0),
                    2
                )

    return output


# Detect potholes
def detect_pothole(frame, gray):

    output = frame.copy()

    blur = cv2.GaussianBlur(gray, (7,7), 0)

    _, th = cv2.threshold(
        blur,
        80,
        255,
        cv2.THRESH_BINARY_INV
    )

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_OPEN,
        np.ones((7,7), np.uint8)
    )

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_CLOSE,
        np.ones((13,13), np.uint8)
    )

    contours, _ = cv2.findContours(
        th,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        area = cv2.contourArea(c)

        if 800 < area < 25000:

            x, y, w, h = cv2.boundingRect(c)

            ratio = w / float(h + 1e-5)

            if 0.5 < ratio < 3.0:

                cv2.rectangle(
                    output,
                    (x,y),
                    (x+w,y+h),
                    (0,0,255),
                    2
                )

                cv2.putText(
                    output,
                    "Pothole",
                    (x,y-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0,0,255),
                    2
                )

    return output


# Merge nearby boxes
def merge_boxes(boxes, distance=50):

    merged_boxes = []

    while boxes:

        x1, y1, x2, y2 = boxes.pop(0)

        merged = True

        while merged:

            merged = False

            remove_indices = []

            for i, (xx1, yy1, xx2, yy2) in enumerate(boxes):

                if (
                    xx1 < x2 + distance and
                    xx2 > x1 - distance and
                    yy1 < y2 + distance and
                    yy2 > y1 - distance
                ):

                    x1 = min(x1, xx1)
                    y1 = min(y1, yy1)
                    x2 = max(x2, xx2)
                    y2 = max(y2, yy2)

                    remove_indices.append(i)

                    merged = True

            for index in sorted(remove_indices, reverse=True):
                boxes.pop(index)

        merged_boxes.append([x1, y1, x2, y2])

    return merged_boxes


# Detect raveling
def detect_raveling(frame, gray):

    output = frame.copy()

    gaussian = cv2.GaussianBlur(
        gray,
        (5,5),
        0
    )

    binary = cv2.adaptiveThreshold(
        gaussian,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        21,
        5
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        np.ones((3,3), np.uint8)
    )

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []

    for c in contours:

        area = cv2.contourArea(c)

        if 800 < area < 5000:

            x, y, w, h = cv2.boundingRect(c)

            boxes.append([
                x,
                y,
                x+w,
                y+h
            ])

    merged_boxes = merge_boxes(boxes)

    for box in merged_boxes:

        x1, y1, x2, y2 = box

        cv2.rectangle(
            output,
            (x1, y1),
            (x2, y2),
            (0,165,255),
            2
        )

        cv2.putText(
            output,
            "Raveling",
            (x1, y1-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0,165,255),
            2
        )

    return output


# Detect transverse cracking
def detect_transverse(frame, gray):

    output = frame.copy()

    blur = cv2.GaussianBlur(
        gray,
        (5,5),
        0
    )

    edges = cv2.Canny(
        blur,
        80,
        180
    )

    kernel = np.ones((3,3), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel
    )

    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi/180,
        threshold=120,
        minLineLength=150,
        maxLineGap=10
    )

    if lines is not None:

        for line in lines:

            x1, y1, x2, y2 = line[0]

            length = np.sqrt(
                (x2 - x1)**2 +
                (y2 - y1)**2
            )

            angle = abs(
                np.degrees(
                    np.arctan2(
                        y2-y1,
                        x2-x1
                    )
                )
            )

            if 80 < angle < 100 and 120 < length < 500:

                cv2.line(
                    output,
                    (x1,y1),
                    (x2,y2),
                    (255,0,255),
                    3
                )

                cv2.putText(
                    output,
                    "Transverse Crack",
                    (x1,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255,0,255),
                    2
                )

    return output


# Read video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))


# Video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)


# Process video
frame_no = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_name = f"{os.path.basename(video_path).split('.')[0]}_{frame_no}.jpg"

    cv2.imwrite(
        os.path.join(extracted_folder, frame_name),
        frame
    )

    enhanced = enhance_image(frame)

    road_mask = create_road_mask(enhanced)

    enhanced = cv2.bitwise_and(
        enhanced,
        road_mask
    )

    cv2.imwrite(
        os.path.join(final_enhanced_folder, frame_name),
        enhanced
    )

    result = frame.copy()

    result = detect_alligator(
        result,
        enhanced
    )

    result = detect_pothole(
        result,
        enhanced
    )

    result = detect_raveling(
        result,
        enhanced
    )

    result = detect_transverse(
        result,
        enhanced
    )

    cv2.imwrite(
        os.path.join(segmented_folder, frame_name),
        result
    )

    cv2.imshow(
        "Road Damage Detection",
        result
    )

    writer.write(result)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    frame_no += 1


# Release
cap.release()

writer.release()

cv2.waitKey(0)

cv2.destroyAllWindows()

print("Road damage detection video created successfully!")

In [ ]:
import cv2
import os
import numpy as np


# Extract frames
def extract_frames(video_path, output_folder):

    cap = cv2.VideoCapture(video_path)

    video_name = os.path.basename(video_path).split(".")[0]

    frame_no = 0

    if not cap.isOpened():
        print("Error opening video")
        return

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_name = f"{video_name}_{frame_no}.jpg"

        cv2.imwrite(
            os.path.join(output_folder, frame_name),
            frame
        )

        frame_no += 1

    cap.release()

    print(f"{frame_no} frames extracted successfully!")


# Video path
video_path = input("Enter video path: ")


# Folder paths
extracted_folder = "../finalRun/frames/extracted"

final_enhanced_folder = "../finalRun/frames/enhanced/4_FinalEnhanced"

segmented_folder = "../finalRun/frames/segmented"

output_video = "../finalRun/final_segmented_video.mp4"


# Create folders
os.makedirs(extracted_folder, exist_ok=True)

os.makedirs(final_enhanced_folder, exist_ok=True)

os.makedirs(segmented_folder, exist_ok=True)


# Extract frames
extract_frames(video_path, extracted_folder)


# Enhance image
def enhance_image(img):

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    median = cv2.medianBlur(gray, 3)

    blur = cv2.GaussianBlur(
        median,
        (5,5),
        0
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    enhanced = clahe.apply(blur)

    kernel = np.array([
        [0,-1,0],
        [-1,5,-1],
        [0,-1,0]
    ])

    sharpened = cv2.filter2D(
        enhanced,
        -1,
        kernel
    )

    return sharpened


# Create road mask
def create_road_mask(gray):

    height, width = gray.shape

    mask = np.zeros_like(gray)

    polygon = np.array([[

        (0, height),

        (width, height),

        (int(width * 0.75), int(height * 0.35)),

        (int(width * 0.25), int(height * 0.35))

    ]], np.int32)

    cv2.fillPoly(
        mask,
        polygon,
        255
    )

    return mask


# Detect alligator cracking
def detect_alligator(frame, gray):

    output = frame.copy()

    blur = cv2.GaussianBlur(gray, (5,5), 0)

    blackhat_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (15,15)
    )

    blackhat = cv2.morphologyEx(
        blur,
        cv2.MORPH_BLACKHAT,
        blackhat_kernel
    )

    _, binary = cv2.threshold(
        blackhat,
        35,
        255,
        cv2.THRESH_BINARY
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        np.ones((7,7), np.uint8)
    )

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        area = cv2.contourArea(c)

        if 3000 < area < 25000:

            x, y, w, h = cv2.boundingRect(c)

            ratio = w / float(h + 1e-5)

            fill_ratio = area / float(w*h)

            if (
                0.7 < ratio < 3.5 and
                fill_ratio > 0.25
            ):

                cv2.rectangle(
                    output,
                    (x,y),
                    (x+w,y+h),
                    (0,255,0),
                    2
                )

                cv2.putText(
                    output,
                    "Alligator Crack",
                    (x,y-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0,255,0),
                    2
                )

    return output


# Detect potholes
def detect_pothole(frame, gray):

    output = frame.copy()

    blur = cv2.GaussianBlur(gray, (9,9), 0)

    _, thresh = cv2.threshold(
        blur,
        70,
        255,
        cv2.THRESH_BINARY_INV
    )

    thresh = cv2.morphologyEx(
        thresh,
        cv2.MORPH_CLOSE,
        np.ones((11,11), np.uint8)
    )

    thresh = cv2.morphologyEx(
        thresh,
        cv2.MORPH_OPEN,
        np.ones((5,5), np.uint8)
    )

    contours, _ = cv2.findContours(
        thresh,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        area = cv2.contourArea(c)

        if 4000 < area < 50000:

            x, y, w, h = cv2.boundingRect(c)

            ratio = w / float(h + 1e-5)

            if 0.5 < ratio < 3.5:

                cv2.rectangle(
                    output,
                    (x,y),
                    (x+w,y+h),
                    (0,0,255),
                    3
                )

                cv2.putText(
                    output,
                    "Pothole",
                    (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0,0,255),
                    2
                )

    return output


# Merge nearby boxes
def merge_boxes(boxes, distance=50):

    merged_boxes = []

    while boxes:

        x1, y1, x2, y2 = boxes.pop(0)

        merged = True

        while merged:

            merged = False

            remove_indices = []

            for i, (xx1, yy1, xx2, yy2) in enumerate(boxes):

                if (
                    xx1 < x2 + distance and
                    xx2 > x1 - distance and
                    yy1 < y2 + distance and
                    yy2 > y1 - distance
                ):

                    x1 = min(x1, xx1)
                    y1 = min(y1, yy1)
                    x2 = max(x2, xx2)
                    y2 = max(y2, yy2)

                    remove_indices.append(i)

                    merged = True

            for index in sorted(remove_indices, reverse=True):
                boxes.pop(index)

        merged_boxes.append([x1, y1, x2, y2])

    return merged_boxes


# Detect raveling
def detect_raveling(frame, gray):

    output = frame.copy()

    gaussian = cv2.GaussianBlur(
        gray,
        (5,5),
        0
    )

    binary = cv2.adaptiveThreshold(
        gaussian,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        21,
        5
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        np.ones((3,3), np.uint8)
    )

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []

    for c in contours:

        area = cv2.contourArea(c)

        if 1500 < area < 8000:

            x, y, w, h = cv2.boundingRect(c)

            ratio = w / float(h + 1e-5)

            if 0.5 < ratio < 4.0:

                boxes.append([
                    x,
                    y,
                    x+w,
                    y+h
                ])

    merged_boxes = merge_boxes(boxes)

    for box in merged_boxes:

        x1, y1, x2, y2 = box

        cv2.rectangle(
            output,
            (x1, y1),
            (x2, y2),
            (0,165,255),
            2
        )

        cv2.putText(
            output,
            "Raveling",
            (x1, y1-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0,165,255),
            2
        )

    return output


# Disable transverse crack detection
def detect_transverse(frame, gray):

    return frame


# Read video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

width = 960
height = 540


# Video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)


# Process video
frame_no = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.resize(
        frame,
        (width, height)
    )

    frame_name = f"{os.path.basename(video_path).split('.')[0]}_{frame_no}.jpg"

    enhanced = enhance_image(frame)

    road_mask = create_road_mask(enhanced)

    enhanced = cv2.bitwise_and(
        enhanced,
        road_mask
    )

    cv2.imwrite(
        os.path.join(final_enhanced_folder, frame_name),
        enhanced
    )

    result = frame.copy()

    result = detect_alligator(
        result,
        enhanced
    )

    result = detect_pothole(
        result,
        enhanced
    )

    result = detect_raveling(
        result,
        enhanced
    )

    result = detect_transverse(
        result,
        enhanced
    )

    cv2.imwrite(
        os.path.join(segmented_folder, frame_name),
        result
    )

    cv2.imshow(
        "Road Damage Detection",
        result
    )

    writer.write(result)

    if cv2.waitKey(40) & 0xFF == ord('q'):
        break

    frame_no += 1


# Release
cap.release()

writer.release()

cv2.destroyAllWindows()

print("Road damage detection video created successfully!")

682 frames extracted successfully!
